# ⚔️ 17. Feature Engineering: RGB + AVG + NTSC Training SVM

Notebook ini melatih Support Vector Machine (SVM) dengan RBF Kernel pada **Combined Feature Vector 1.536-dimensi** hasil Feature Fusion Tahap 16.
- **Anti Data Leakage**: StandardScaler hanya di-fit pada data training.
- **Tujuan**: Memanfaatkan sinergi representasi warna RGB dan karakteristik luminansi Grayscale untuk mencapai akurasi optimal (>=95%).
- **Target Artifact**: cifar10_artifacts/rgb_avg_ntsc/scaler.pkl & svm_model.pkl.


In [1]:
import os
import sys
import pickle
import json
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ARTIFACT_DIR = 'cifar10_artifacts/rgb_avg_ntsc'
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
print(f"Memuat vektor fitur gabungan training dari: {train_feat_path}")
train_data = np.load(train_feat_path)
X_train_features = train_data['X_train_features']
y_train = train_data['y_train']
print(f"Dimensi fitur gabungan: {X_train_features.shape}")

RANDOM_STATE = 23092026

Memuat vektor fitur gabungan training dari: cifar10_artifacts/rgb_avg_ntsc\train_features.npz


Dimensi fitur gabungan: (50000, 1536)


## 🛠️ 1. Standarisasi Fitur & Melatih Model SVM Classifier pada 1.536-Dimensi Fitur Gabungan


In [2]:
# 1. Standarisasi Fitur Gabungan Training (1.536 dimensi)
scaler = StandardScaler()
print("Menstandarisasi vektor fitur gabungan training dengan StandardScaler...")
X_train_scaled = scaler.fit_transform(X_train_features)
print(f"[BERHASIL] Fitur gabungan telah distandarisasi. Shape: {X_train_scaled.shape}")

print()
print("2. Melatih SVM RBF Classifier pada seluruh vektor fitur gabungan (50.000 sampel)...")
svm_model = SVC(
    C=10.0,
    gamma='scale',
    kernel='rbf',
    cache_size=4096,
    random_state=RANDOM_STATE,
    decision_function_shape='ovr'
)
svm_model.fit(X_train_scaled, y_train)
print("[BERHASIL] SVM Fused Classifier telah selesai dilatih!")

sample_acc = accuracy_score(y_train[:5000], svm_model.predict(X_train_scaled[:5000]))
print(f"Akurasi SVM Fused pada sampel training: {sample_acc * 100:.2f}%")

Menstandarisasi vektor fitur gabungan training dengan StandardScaler...


[BERHASIL] Fitur gabungan telah distandarisasi. Shape: (50000, 1536)

2. Melatih SVM RBF Classifier pada seluruh vektor fitur gabungan (50.000 sampel)...


[BERHASIL] SVM Fused Classifier telah selesai dilatih!


Akurasi SVM Fused pada sampel training: 99.68%


## 📦 2. Menyimpan Scaler, Model SVM & Metadata ke Disk


In [3]:
scaler_save_path = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
svm_save_path    = os.path.join(ARTIFACT_DIR, 'svm_model.pkl')
meta_save_path   = os.path.join(ARTIFACT_DIR, 'metadata.json')

with open(scaler_save_path, 'wb') as f:
    pickle.dump(scaler, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(svm_save_path, 'wb') as f:
    pickle.dump(svm_model, f, protocol=pickle.HIGHEST_PROTOCOL)

metadata = {
    'method': 'Feature Engineering (RGB + AVG + NTSC) + SVM',
    'fused_components': ['RGB (512)', 'Grayscale AVG (512)', 'Grayscale NTSC (512)'],
    'svm_C': 10.0,
    'svm_gamma': 'scale',
    'total_feature_dim': int(X_train_features.shape[1])
}

with open(meta_save_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"[BERHASIL] Scaler tersimpan ke: {scaler_save_path}")
print(f"[BERHASIL] SVM Model tersimpan ke: {svm_save_path}")
print(f"[BERHASIL] Metadata tersimpan ke: {meta_save_path}")

[BERHASIL] Scaler tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\scaler.pkl
[BERHASIL] SVM Model tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\svm_model.pkl
[BERHASIL] Metadata tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\metadata.json
